In [ ]:
import pandas as pd
from gurobipy import *

# 从Excel读取数据
excel_file = 'C:\\Users\\user\\Desktop\\論\\data.xlsx'

df_dmu = pd.read_excel(excel_file, sheet_name='Sheet1')
df_x = pd.read_excel(excel_file, sheet_name='Sheet2')
df_z = pd.read_excel(excel_file, sheet_name='Sheet3')
df_y = pd.read_excel(excel_file, sheet_name='Sheet4')

DMU = df_dmu['DMU'].tolist()

x = {row['DMU']: row[['I1']].tolist() for idx, row in df_x.iterrows()}
z = {row['DMU']: row[['G2']].tolist() for idx, row in df_z.iterrows()}
zb1 = {row['DMU']: row[['G1']].tolist() for idx, row in df_z.iterrows()}
zb2 = {row['DMU']: row[['G3']].tolist() for idx, row in df_z.iterrows()}
y = {row['DMU']: row[['O2']].tolist() for idx, row in df_y.iterrows()}
yb1 = {row['DMU']: row[['O1']].tolist() for idx, row in df_y.iterrows()}
yb2 = {row['DMU']: row[['O3']].tolist() for idx, row in df_y.iterrows()}

I = 1
G = 1
O = 1
BG = 1
BO = 1

first_stage_efficiencies = {}

for k0 in DMU:
    m = Model("network_DEA")

    v = {k: m.addVar(vtype=GRB.CONTINUOUS, name="v_%s" % k, lb=0) for k in DMU}
    u = {k: m.addVar(vtype=GRB.CONTINUOUS, name="v_%s" % k, lb=0) for k in DMU}
    a = m.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY, name="a")
    b = m.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY, name="b")
    # 设置目标函数
    m.setObjective(a+b, GRB.MAXIMIZE)

    # 添加约束
    for i in range(I):
        m.addConstr(quicksum(v[k] * x[k][i] for k in DMU) <= x[k0][i])
    for g in range(G):
        m.addConstr(quicksum(v[k] * z[k][g] for k in DMU) >= z[k0][g] * a)
    for bg in range(BG):
        m.addConstr(quicksum(v[k] * zb2[k][bg] for k in DMU) <= zb2[k0][bg] * a)
        m.addConstr(quicksum(v[k] * zb1[k][bg] for k in DMU) >= zb1[k0][bg] * a)
    m.addConstr(quicksum(v[k] for k in DMU) == 1)
    for g in range(G):
        m.addConstr(quicksum(u[k] * z[k][g] for k in DMU) <= z[k0][g])
    for bg in range(BG):
        m.addConstr(quicksum(u[k] * zb2[k][bg] for k in DMU) >= zb2[k0][bg] )
        m.addConstr(quicksum(u[k] * zb1[k][bg] for k in DMU) <= zb1[k0][bg] )
    for o in range(O):
        m.addConstr(quicksum(u[k] * y[k][o] for k in DMU) >= y[k0][o])
    for bo in range(BO):
        m.addConstr(quicksum(u[k] * yb2[k][bo] for k in DMU) <= yb2[k0][bo])
        m.addConstr(quicksum(u[k] * yb1[k][bo] for k in DMU) >= yb1[k0][bo])
    m.addConstr(quicksum(u[k] for k in DMU) == 1)
    # 优化模型
    m.optimize()

    # 存储每个DMU的效率值到first_stage_efficiencies字典中
    if m.status == GRB.OPTIMAL:
        first_stage_efficiencies[k0] = a.X
    else:
        first_stage_efficiencies[k0] = None

    # 调试语句，确认每个DMU的效率值是否正确计算
    print("DMU %s 的效率为：%0.3g" % (k0, a.X))

# 打印每个DMU的效率值
for k in first_stage_efficiencies:
    print("DMU %s 的效率为：%0.3g" % (k, first_stage_efficiencies[k]))

Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R) Core(TM) i3-8100 CPU @ 3.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 4 logical processors, using up to 4 threads

Optimize a model with 12 rows, 22 columns and 94 nonzeros
Model fingerprint: 0xa10f21d3
Coefficient statistics:
  Matrix range     [1e+00, 5e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 2e+01]
Presolve removed 5 rows and 10 columns
Presolve time: 0.01s
Presolved: 7 rows, 12 columns, 35 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    3.5515152e+00   9.298633e+00   0.000000e+00      0s
       5    2.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 5 iterations and 0.01 seconds (0.00 work units)
Optimal objective  2.000000000e+00
DMU DMMU1 的效率为：1
Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R

Thread count: 4 physical cores, 4 logical processors, using up to 4 threads

Optimize a model with 12 rows, 22 columns and 94 nonzeros
Model fingerprint: 0xef65fd02
Coefficient statistics:
  Matrix range     [1e+00, 5e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 3e+01]
Presolve removed 6 rows and 11 columns
Presolve time: 0.01s
Presolved: 6 rows, 11 columns, 30 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    3.0000000e+00   2.907000e+00   0.000000e+00      0s
       7    2.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 7 iterations and 0.01 seconds (0.00 work units)
Optimal objective  2.000000000e+00
DMU DMU10 的效率为：1
DMU DMMU1 的效率为：1
DMU DMMU2 的效率为：1.41
DMU DMMU3 的效率为：1
DMU DMU4 的效率为：1
DMU DMU5 的效率为：1
DMU DMU6 的效率为：1
DMU DMMU7 的效率为：3.94
DMU DMU8 的效率为：1.14
DMU DMU9 的效率为：1.1
DMU DMU10 的效率为：1


In [ ]:
!pip install gurobipy -q

import pandas as pd
from gurobipy import *
from google.colab import files

# 上傳 DATARUN.xlsx
uploaded = files.upload()

excel_file = list(uploaded.keys())[0]

# 重新讀取
df_dmu = pd.read_excel(excel_file, sheet_name='Sheet1')
df_x   = pd.read_excel(excel_file, sheet_name='Sheet2')
df_z   = pd.read_excel(excel_file, sheet_name='Sheet3')
df_y   = pd.read_excel(excel_file, sheet_name='Sheet4')

# 清除可能的欄位空白
for df in [df_dmu, df_x, df_z, df_y]:
    df.columns = df.columns.astype(str).str.strip()

# 檢查
print("Sheet1:", df_dmu.columns.tolist())
print("Sheet2:", df_x.columns.tolist())
print("Sheet3:", df_z.columns.tolist())
print("Sheet4:", df_y.columns.tolist())

# 建立 DMU
DMU = df_dmu['DMU'].tolist()

print("DMU 數量：", len(DMU))
print("前 10 筆：", DMU[:10])

Saving DRALL11.xlsx to DRALL11.xlsx
Sheet1: ['DMU']
Sheet2: ['DMU', 'I1', 'I2']
Sheet3: ['DMU', 'G1', 'G2', 'G3']
Sheet4: ['DMU', 'O1', 'O2', 'O3', 'O4']
DMU 數量： 403
前 10 筆： [1101, 1102, 1103, 1104, 1201, 1210, 1215, 1216, 1217, 1218]


In [ ]:
#這個才對
import pandas as pd
from gurobipy import *
from google.colab import files


def run_dea(excel_file, Sheet, output_file='DEA_result.xlsx', download=True):
    df_dmu = pd.read_excel(excel_file, sheet_name= Sheet)



    DMU = df_dmu['DMU'].tolist()

    x = {row['DMU']: row[['I1','I2']].tolist() for idx, row in df_dmu.iterrows()}
    z = {row['DMU']: row[['G3']].tolist() for idx, row in df_dmu.iterrows()}
    zb1 = {row['DMU']: row[['G1']].tolist() for idx, row in df_dmu.iterrows()}
    zb2 = {row['DMU']: row[['G2']].tolist() for idx, row in df_dmu.iterrows()}
    y = {row['DMU']: row[['O3','O4']].tolist() for idx, row in df_dmu.iterrows()}
    yb1 = {row['DMU']: row[['O1']].tolist() for idx, row in df_dmu.iterrows()}
    yb2 = {row['DMU']: row[['O2']].tolist() for idx, row in df_dmu.iterrows()}

    I = 2
    G = 1
    O = 2
    BG = 1
    BO = 1

    first_stage_efficiencies = {}
    second_stage_efficiencies = {}
    for k0 in DMU:
        m = Model("network_DEA")

        v = {k: m.addVar(vtype=GRB.CONTINUOUS, name="v_%s" % k, lb=0) for k in DMU}
        u = {k: m.addVar(vtype=GRB.CONTINUOUS, name="u_%s" % k, lb=0) for k in DMU}
        a = m.addVar(lb=0, ub=GRB.INFINITY, name="a")
        b = m.addVar(lb=0, ub=GRB.INFINITY, name="b")
        c = m.addVar(lb=0, ub=GRB.INFINITY, name="c")
        c1 = m.addVar(lb=0, ub=GRB.INFINITY, name="c1")
        c2 = m.addVar(lb=0, ub=GRB.INFINITY, name="c2")
        w1 = m.addVar(lb=0, ub=GRB.INFINITY, name="w1")
        w2 = m.addVar(lb=0, ub=GRB.INFINITY, name="w2")

        # 设置目标函数：max a
        m.setObjective(a-b, GRB.MINIMIZE)

        # 添加约束
        for i in range(I):
            m.addConstr(quicksum(v[k] * x[k][i] for k in DMU) <= x[k0][i]*a)
        for g in range(G):
            m.addConstr(quicksum(v[k] * z[k][g] for k in DMU) >= c)
        for bg in range(BG):
            m.addConstr(quicksum(v[k] * zb2[k][bg] for k in DMU) <= c2)
            m.addConstr(quicksum(v[k] * zb1[k][bg] for k in DMU) >= c1)
        m.addConstr(quicksum(v[k] for k in DMU) == 1)

        # 第二阶段约束
        for g in range(G):
            m.addConstr(quicksum(u[k] * z[k][g] for k in DMU) <= c)
            #m.addConstr(quicksum(u[k] * z[k][g] for k in DMU) <= quicksum(v[k] * z[k][g] for k in DMU))
        for bg in range(BG):
            m.addConstr(quicksum(u[k] * zb2[k][bg] for k in DMU) >= c2)
            m.addConstr(quicksum(u[k] * zb1[k][bg] for k in DMU) <= c1)
            #m.addConstr(quicksum(u[k] * zb1[k][bg] for k in DMU) <= quicksum(v[k] * zb1[k][bg] for k in DMU))
            #m.addConstr(quicksum(u[k] * zb2[k][bg] for k in DMU) >= quicksum(v[k] * zb2[k][bg] for k in DMU))
        for o in range(O):
            m.addConstr(quicksum(u[k] * y[k][o] for k in DMU) >= y[k0][o] * b)
        for bo in range(BO):
            m.addConstr(quicksum(u[k] * yb2[k][bo] for k in DMU) <= yb2[k0][bo] * b)
            m.addConstr(quicksum(u[k] * yb1[k][bo] for k in DMU) >= yb1[k0][bo] * b)
        m.addConstr(quicksum(u[k] for k in DMU) == 1)
        m.addConstr(b >= 1)
        m.addConstr(a <= 1)
        #m.addConstr(w1+w2 == 1)

        # 优化模型
        m.optimize()

        if m.status == GRB.OPTIMAL:
            first_stage_efficiencies[k0] = a.X
            second_stage_efficiencies[k0] = b.X
        else:
            first_stage_efficiencies[k0] = None
            second_stage_efficiencies[k0] = None

        # 调试语句，确认每个DMU的效率值是否正确计算
        print("DMU %s 的第一阶段效率为：%0.3g" % (k0, a.X))
        print("DMU %s 的第二阶段效率为：%0.3g" % (k0, b.X))

    # 打印每个DMU的效率值
    print("第一阶段效率：")
    for k in first_stage_efficiencies:
        print("DMU %s 的第一阶段效率为：%0.3g" % (k, first_stage_efficiencies[k]))

    print("第二阶段效率：")
    for k in second_stage_efficiencies:
        print("DMU %s 的第二阶段效率为：%0.3g" % (k, second_stage_efficiencies[k]))
    result_df = pd.DataFrame({
        Sheet + '_DMU': DMU,
        Sheet + '_第一階段效率': [first_stage_efficiencies[k] for k in DMU],
        Sheet + '_第二階段效率': [second_stage_efficiencies[k] for k in DMU],
        Sheet + '_總效率': [second_stage_efficiencies[k]*first_stage_efficiencies[k] for k in DMU]
    })

    return result_df

In [ ]:
!pip install gurobipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 32.0 MB/s eta 0:00:00


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving adrun1.xlsx to adrun1.xlsx


In [ ]:
#這個才對無ESG
import pandas as pd
from gurobipy import *
from google.colab import files
import os
# 从Excel读取数据
#excel_file = 'C:\\Users\\user\\Desktop\\論\\data.xlsx'
excel_file = 'DRALLTRA.xlsx'
result_df = pd.DataFrame()
def run_dea1(excel_file, Sheet, output_file='DEA_result.xlsx', download=True):
    file_name = os.path.splitext(os.path.basename(excel_file))[0]
    df_dmu = pd.read_excel(excel_file, sheet_name= Sheet)

    DMU = df_dmu['DMU'].tolist()

    x = {row['DMU']: row[['I1','I2']].tolist() for idx, row in df_dmu.iterrows()}
    #z = {row['DMU']: row[['G3']].tolist() for idx, row in df_z.iterrows()}
    zb1 = {row['DMU']: row[['G1']].tolist() for idx, row in df_dmu.iterrows()}
    zb2 = {row['DMU']: row[['G2']].tolist() for idx, row in df_dmu.iterrows()}
    y = {row['DMU']: row[['O3','O4']].tolist() for idx, row in df_dmu.iterrows()}
    yb1 = {row['DMU']: row[['O1']].tolist() for idx, row in df_dmu.iterrows()}
    yb2 = {row['DMU']: row[['O2']].tolist() for idx, row in df_dmu.iterrows()}

    I = 2
    G = 0
    O = 2
    BG = 1
    BO = 1

    first_stage_efficiencies = {}
    second_stage_efficiencies = {}
    for k0 in DMU:
        m = Model("network_DEA")

        v = {k: m.addVar(vtype=GRB.CONTINUOUS, name="v_%s" % k, lb=0) for k in DMU}
        u = {k: m.addVar(vtype=GRB.CONTINUOUS, name="u_%s" % k, lb=0) for k in DMU}
        a = m.addVar(lb=0, ub=GRB.INFINITY, name="a")
        b = m.addVar(lb=0, ub=GRB.INFINITY, name="b")
        c = m.addVar(lb=0, ub=GRB.INFINITY, name="c")
        c1 = m.addVar(lb=0, ub=GRB.INFINITY, name="c1")
        c2 = m.addVar(lb=0, ub=GRB.INFINITY, name="c2")
        w1 = m.addVar(lb=0, ub=GRB.INFINITY, name="w1")
        w2 = m.addVar(lb=0, ub=GRB.INFINITY, name="w2")

        # 设置目标函数：max a
        m.setObjective(a-b, GRB.MINIMIZE)

        # 添加约束
        for i in range(I):
            m.addConstr(quicksum(v[k] * x[k][i] for k in DMU) <= x[k0][i]*a)
        #for g in range(G):
            #m.addConstr(quicksum(v[k] * z[k][g] for k in DMU) >= c)
        for bg in range(BG):
            m.addConstr(quicksum(v[k] * zb2[k][bg] for k in DMU) <= c2)
            m.addConstr(quicksum(v[k] * zb1[k][bg] for k in DMU) >= c1)
        m.addConstr(quicksum(v[k] for k in DMU) == 1)

        # 第二阶段约束
      # for g in range(G):
          # m.addConstr(quicksum(u[k] * z[k][g] for k in DMU) <= c)
            #m.addConstr(quicksum(u[k] * z[k][g] for k in DMU) <= quicksum(v[k] * z[k][g] for k in DMU))
        for bg in range(BG):
            m.addConstr(quicksum(u[k] * zb2[k][bg] for k in DMU) >= c2)
            m.addConstr(quicksum(u[k] * zb1[k][bg] for k in DMU) <= c1)
            #m.addConstr(quicksum(u[k] * zb1[k][bg] for k in DMU) <= quicksum(v[k] * zb1[k][bg] for k in DMU))
            #m.addConstr(quicksum(u[k] * zb2[k][bg] for k in DMU) >= quicksum(v[k] * zb2[k][bg] for k in DMU))
        for o in range(O):
            m.addConstr(quicksum(u[k] * y[k][o] for k in DMU) >= y[k0][o] * b)
        for bo in range(BO):
            m.addConstr(quicksum(u[k] * yb2[k][bo] for k in DMU) <= yb2[k0][bo] * b)
            m.addConstr(quicksum(u[k] * yb1[k][bo] for k in DMU) >= yb1[k0][bo] * b)
        m.addConstr(quicksum(u[k] for k in DMU) == 1)
        m.addConstr(b >= 1)
        m.addConstr(a <= 1)
        #m.addConstr(w1+w2 == 1)

        # 优化模型
        m.optimize()

        if m.status == GRB.OPTIMAL:
            first_stage_efficiencies[k0] = a.X
            second_stage_efficiencies[k0] = b.X
        else:
            first_stage_efficiencies[k0] = None
            second_stage_efficiencies[k0] = None

        # 调试语句，确认每个DMU的效率值是否正确计算
        print("DMU %s 的第一阶段效率为：%0.3g" % (k0, a.X))
        print("DMU %s 的第二阶段效率为：%0.3g" % (k0, b.X))

    # 打印每个DMU的效率值
    print("第一阶段效率：")
    for k in first_stage_efficiencies:
        print("DMU %s 的第一阶段效率为：%0.3g" % (k, first_stage_efficiencies[k]))

    print("第二阶段效率：")
    for k in second_stage_efficiencies:
        print("DMU %s 的第二阶段效率为：%0.3g" % (k, second_stage_efficiencies[k]))
    result_df = pd.DataFrame({
        Sheet + '_DMU': DMU,
        Sheet + '_第一階段效率': [first_stage_efficiencies[k] for k in DMU],
        Sheet + '_第二階段效率': [second_stage_efficiencies[k] for k in DMU],
        Sheet + '_總效率': [second_stage_efficiencies[k]*first_stage_efficiencies[k] for k in DMU]
    })

    return result_df

In [ ]:
#這個才對 NEW
import pandas as pd
from gurobipy import *
from google.colab import files


def run_dea2(excel_file, Sheet, output_file='DEA_result.xlsx', download=True):
    df_dmu = pd.read_excel(excel_file, sheet_name= Sheet)



    DMU = df_dmu['DMU'].tolist()

    x = {row['DMU']: row[['I1','I2']].tolist() for idx, row in df_dmu.iterrows()}
    #z = {row['DMU']: row[['G3']].tolist() for idx, row in df_dmu.iterrows()}
    zb1 = {row['DMU']: row[['G1']].tolist() for idx, row in df_dmu.iterrows()}
    zb2 = {row['DMU']: row[['G2']].tolist() for idx, row in df_dmu.iterrows()}
    y = {row['DMU']: row[['O3','O4','O1']].tolist() for idx, row in df_dmu.iterrows()}
    #yb1 = {row['DMU']: row[['O1']].tolist() for idx, row in df_dmu.iterrows()}
    #yb2 = {row['DMU']: row[['O2']].tolist() for idx, row in df_dmu.iterrows()}

    I = 2
    G = 1
    O = 2
    BG = 1
    BO = 1

    first_stage_efficiencies = {}
    second_stage_efficiencies = {}
    for k0 in DMU:
        m = Model("network_DEA")

        v = {k: m.addVar(vtype=GRB.CONTINUOUS, name="v_%s" % k, lb=0) for k in DMU}
        u = {k: m.addVar(vtype=GRB.CONTINUOUS, name="u_%s" % k, lb=0) for k in DMU}
        a = m.addVar(lb=0, ub=GRB.INFINITY, name="a")
        b = m.addVar(lb=0, ub=GRB.INFINITY, name="b")
        c = m.addVar(lb=0, ub=GRB.INFINITY, name="c")
        c1 = m.addVar(lb=0, ub=GRB.INFINITY, name="c1")
        c2 = m.addVar(lb=0, ub=GRB.INFINITY, name="c2")
        w1 = m.addVar(lb=0, ub=GRB.INFINITY, name="w1")
        w2 = m.addVar(lb=0, ub=GRB.INFINITY, name="w2")

        # 设置目标函数：max a
        m.setObjective(a-b, GRB.MINIMIZE)

        # 添加约束
        for i in range(I):
            m.addConstr(quicksum(v[k] * x[k][i] for k in DMU) <= x[k0][i]*a)
        #for g in range(G):
            #m.addConstr(quicksum(v[k] * z[k][g] for k in DMU) >= c)
        for bg in range(BG):
            m.addConstr(quicksum(v[k] * zb2[k][bg] for k in DMU) <= c2)
            m.addConstr(quicksum(v[k] * zb1[k][bg] for k in DMU) >= c1)
        m.addConstr(quicksum(v[k] for k in DMU) == 1)

        # 第二阶段约束
        #for g in range(G):
            #m.addConstr(quicksum(u[k] * z[k][g] for k in DMU) <= c)
            #m.addConstr(quicksum(u[k] * z[k][g] for k in DMU) <= quicksum(v[k] * z[k][g] for k in DMU))
        for bg in range(BG):
            m.addConstr(quicksum(u[k] * zb2[k][bg] for k in DMU) >= c2)
            m.addConstr(quicksum(u[k] * zb1[k][bg] for k in DMU) <= c1)
            #m.addConstr(quicksum(u[k] * zb1[k][bg] for k in DMU) <= quicksum(v[k] * zb1[k][bg] for k in DMU))
            #m.addConstr(quicksum(u[k] * zb2[k][bg] for k in DMU) >= quicksum(v[k] * zb2[k][bg] for k in DMU))
        for o in range(O):
            m.addConstr(quicksum(u[k] * y[k][o] for k in DMU) >= y[k0][o] * b)
        #for bo in range(BO):
            #m.addConstr(quicksum(u[k] * yb2[k][bo] for k in DMU) <= yb2[k0][bo] * b)
            #m.addConstr(quicksum(u[k] * yb1[k][bo] for k in DMU) >= yb1[k0][bo] * b)
        m.addConstr(quicksum(u[k] for k in DMU) == 1)
        m.addConstr(b >= 1)
        m.addConstr(a <= 1)
        #m.addConstr(w1+w2 == 1)

        # 优化模型
        m.optimize()

        if m.status == GRB.OPTIMAL:
            first_stage_efficiencies[k0] = a.X
            second_stage_efficiencies[k0] = b.X
        else:
            first_stage_efficiencies[k0] = None
            second_stage_efficiencies[k0] = None

        # 调试语句，确认每个DMU的效率值是否正确计算
        print("DMU %s 的第一阶段效率为：%0.3g" % (k0, a.X))
        print("DMU %s 的第二阶段效率为：%0.3g" % (k0, b.X))

    # 打印每个DMU的效率值
    print("第一阶段效率：")
    for k in first_stage_efficiencies:
        print("DMU %s 的第一阶段效率为：%0.3g" % (k, first_stage_efficiencies[k]))

    print("第二阶段效率：")
    for k in second_stage_efficiencies:
        print("DMU %s 的第二阶段效率为：%0.3g" % (k, second_stage_efficiencies[k]))
    result_df = pd.DataFrame({
        Sheet + '_DMU': DMU,
        Sheet + '_第一階段效率': [first_stage_efficiencies[k] for k in DMU],
        Sheet + '_第二階段效率': [second_stage_efficiencies[k] for k in DMU],
        Sheet + '_總效率': [second_stage_efficiencies[k]*first_stage_efficiencies[k] for k in DMU]
    })

    return result_df

In [ ]:
import pandas as pd
import os

# 原始 Excel
excel_file = 'adrun1.xlsx'

# 要逐次排除的 DMU
#exclude_dmus = ["nesg", "esg", "科技", "日常", "房", "工業", "非日常", "醫", "原物料", "100以上", "100到50", "50到20", "20以下"]
exclude_dmus = ["esg"]
# 用來儲存每次 DEA 結果
all_results = []

# 逐一排除
for exclude_dmu in exclude_dmus:

    print(f"\n==============================")
    print(f"本次排除 DMU：{exclude_dmu}")
    print(f"==============================")

    # 暫存檔名
    #temp_file = f'{exclude_dmu}.xlsx'
    #file_name = os.path.splitext(os.path.basename(temp_file))[0]

    # 每次都重新讀原始 Excel
    #sheets = pd.read_excel(
    #    excel_file,
    #    sheet_name=None
    #)

    # 每個 sheet 都刪除本次指定的 DMU

    # with pd.ExcelWriter(
    #     temp_file,
    #     engine='openpyxl'
    # ) as writer:

    #     for sheet_name, df in sheets.items():

    #         df_new = df[
    #             df['DMU'] != exclude_dmu
    #         ].copy()

    #         df_new.to_excel(
    #             writer,
    #             sheet_name=sheet_name,
    #             index=False
    #         )

    # 跑 DEA，接收 result_df
    result_df = run_dea2(excel_file, exclude_dmu)

    # 儲存此次結果
    all_results.append(result_df)


# ============================================================
# 5 次全部跑完後，再左右合併
# ============================================================
final_result = pd.concat(
    all_results,
    axis=1
)

# 存到同一個 Excel
final_result.to_excel(
    'DEA_All1.xlsx',
    index=False
)

print("完成：DEA_結果.xlsx")

串流輸出內容已截斷至最後 5000 行。
DMU 3450 的第二阶段效率为：1.34
Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (linux64 - "Ubuntu 24.04.4 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 12 rows, 825 columns and 3282 nonzeros (Min)
Model fingerprint: 0x0af14e19
Model has 2 linear objective coefficients
Coefficient statistics:
  Matrix range     [2e-01, 6e+09]
  Objective range  [1e+00, 1e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]
         Consider reformulating model or setting NumericFocus parameter
         to avoid numerical issues.

Presolve removed 4 rows and 5 columns
Presolve time: 0.01s
Presolved: 8 rows, 820 columns, 3276 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0   -1.2500000e+29   1.861007e+30   1.250000e-01      0s
      32   -9.5557665e-01   0.000000e+00   0.000000e+00      0s

Solved in 32 i

In [ ]:
from google.colab import files
files.download('DEA_All1.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
 # 輸出檔名
output_file = '無ESG_DEA結果.xlsx'

    # 寫入 Excel
result_df.to_excel(output_file, index=False)

print(f"\n結果已輸出：{output_file}")

    # Colab 自動下載
files.download(output_file)


結果已輸出：無ESG_DEA結果.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
result_df = pd.DataFrame({
    'DMU': DMU,
    '第一階段效率': [first_stage_efficiencies[k] for k in DMU],
    '第二階段效率': [second_stage_efficiencies[k] for k in DMU]
})

# 輸出檔名
output_file = '無ESG_DEA結果.xlsx'

# 寫入 Excel
result_df.to_excel(output_file, index=False)

print(f"\n結果已輸出：{output_file}")

# Colab 自動下載
files.download(output_file)


結果已輸出：無ESG_DEA結果.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>